In [ ]:
app_fqdn = "mtg.test"
proxy_address = "127.0.0.1"
tenant_identifier = "sample"
owner_username = "owner"
owner_password = "E2e-Owner-Pass1"
member_display_name = "Member"
member_username = "member"
member_password = "E2e-Member-Pass1"
room_title = "E2E Room"
default_result_path = None
close_on_fail = False
transition_timeout = 10000

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# WebMeeting E2E Test - Tenant Management

- Test data to prepare: The tenant tenant_identifier and its owner account, created by 01_Admin_Site.ipynb

In [ ]:
import importlib

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *

await init_pw_context(
    close_on_fail=close_on_fail,
    last_path=default_result_path,
    ignore_https_errors=True,
    browser_args=[f"--host-resolver-rules=MAP {app_fqdn} {proxy_address}, MAP *.{app_fqdn} {proxy_address}"],
)

## Log in to the tenant as the owner, and verify that the home page is shown.

In [ ]:
tenant_url = f"https://{tenant_identifier}.{app_fqdn}"

async def _step(page):
    await page.goto(f"{tenant_url}/login")
    await page.locator("input[name=username]").fill(owner_username)
    await page.locator("input[name=password]").fill(owner_password)
    await page.locator("button[type=submit]").click()

    await expect(page).to_have_url(f"{tenant_url}/home", timeout=transition_timeout)

await run_pw(_step)

## Open the user management from the menu, and create the account member_username.

The user list must show the created account.

In [ ]:
async def _step(page):
    await page.get_by_role("link", name="ユーザー管理").click()
    await expect(page).to_have_url(f"{tenant_url}/account", timeout=transition_timeout)
    await page.get_by_role("link", name="ユーザー新規登録").click()

    await page.locator("#displayName").fill(member_display_name)
    await page.locator("#username").fill(member_username)
    await page.locator("#password").fill(member_password)
    await page.locator("#passwordConfirm").fill(member_password)
    await page.locator("button[type=submit]").click()

    await expect(page).to_have_url(f"{tenant_url}/account", timeout=transition_timeout)
    await expect(page.locator("#accountList")).to_contain_text(member_username)

await run_pw(_step)

## Open the room management from the menu, and create the room room_title that any user of the tenant can enter without the permission of the host.

The room detail must show the title and that any user can enter.

In [ ]:
import re

async def _step(page):
    await page.get_by_role("link", name="ルーム管理").click()
    await expect(page).to_have_url(f"{tenant_url}/room", timeout=transition_timeout)
    await page.get_by_role("link", name="新規作成").click()

    await page.locator("#title").fill(room_title)
    await page.locator("#allowAnyUser").check()
    await page.locator("#anyUserPermitRequired").uncheck()
    await page.locator("#submitButton").click()

    await expect(page).to_have_url(re.compile(f"^{tenant_url}/room/[0-9]+$"), timeout=transition_timeout)
    await expect(page.locator("#main")).to_contain_text(room_title)
    allow_any_user = page.locator(".meeting-def-title", has_text="ルームに設定された参加者以外のユーザーの参加")
    await expect(allow_any_user.locator("xpath=following-sibling::div")).to_have_text("可")

await run_pw(_step)

Clean up

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}